# 01. INT, UINT, Fixed-Point (Q-Format), Dyadic Rescaling, and RNS Arithmetic

- **Category:** 14_quantisation / 01_foundations_formats_and_observers
- **Lineage & Citations:** Garner (1959, *The Residue Number System*); Courbariaux et al. (2014, *Training Deep Neural Networks with Low Precision Multiplications*); Jacob et al. (CVPR 2018, *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*).
- **Purpose:** Formulate uniform integer quantization (affine asymmetric vs symmetric), fixed-point $Q_{m.n}$ arithmetic, integer-only dyadic multiplier rescaling, and carry-free Residue Number System (RNS) modular arithmetic.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. Stochastic Rounding and Master Weight Free BF16 Training](../../13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. Logarithmic PoT APoT Mu-Law and Posit Quantization](02_logarithmic_pot_apot_mulaw_and_posit_quantization.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use symmetric quantization on strictly non-negative post-ReLU/SiLU activation distributions (allocating negative bins wastes $50\%$ of all quantization levels, losing $1.0\text{ bit}$ / $\approx 6\text{ dB SQNR}$); do NOT use RNS without saturating dynamic range checks (integer overflow beyond $\prod m_i$ wraps around modulo $M$).

## Mathematical Foundations & Integer Arithmetic Formulations

1. **Affine (Asymmetric) Quantization**:
   $$s = \frac{x_{\max} - x_{\min}}{2^b - 1}, \quad z = \operatorname{round}\left(-\frac{x_{\min}}{s}\right), \quad q = \operatorname{clamp}\left(\operatorname{round}\left(\frac{x}{s}\right) + z, \; 0, \; 2^b - 1\right), \quad \hat{x} = (q - z) s$$
2. **Symmetric Quantization**:
   $$s = \frac{\max(|x|)}{q_{\max}}, \quad q = \operatorname{clamp}\left(\operatorname{round}\left(\frac{x}{s}\right), \; -q_{\max}, \; q_{\max}\right), \quad \hat{x} = q \cdot s$$
3. **Dyadic Multiplier Rescaling**:
   $$M = \frac{s_W s_X}{s_Y} \approx M_0 \cdot 2^{-n}, \quad M_0 \in [2^{30}, 2^{31} - 1], \quad \operatorname{Rescale}(A) = \left\lfloor \frac{A M_0 + 2^{30}}{2^{31}} \right\rfloor \gg (n - 31)$$
4. **Residue Number System (RNS) & Chinese Remainder Theorem (CRT)**:
   $$(a \cdot b) \leftrightarrow ((a \bmod m_i) \cdot (b \bmod m_i) \bmod m_i), \quad X = \left(\sum_{i=1}^k x_i M_i y_i\right) \bmod M$$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def affine_quantize_dequantize(x, num_bits=8):
    qmax = float((1 << num_bits) - 1)
    xmin = float(torch.min(x))
    xmax = float(torch.max(x))
    scale = max(1e-8, (xmax - xmin) / qmax)
    zero_point = float(torch.clamp(torch.round(torch.tensor(-xmin / scale)), 0.0, qmax))
    q = torch.clamp(torch.round(x * (1.0 / scale)) + zero_point, 0.0, qmax)
    x_hat = (q - zero_point) * scale
    return x_hat, q, scale, zero_point

def symmetric_quantize_dequantize(x, num_bits=8, full_range=False):
    qmax = float(1 << (num_bits - 1)) - 1.0
    qmin = -float(1 << (num_bits - 1)) if full_range else -qmax
    amax = float(torch.max(torch.abs(x)))
    scale = max(1e-8, amax / qmax)
    q = torch.clamp(torch.round(x * (1.0 / scale)), qmin, qmax)
    x_hat = q * scale
    return x_hat, q, scale

def fixed_point_qmn(x, m=4, n=4):
    scale = float(1 << n)
    qmax = float(1 << (m + n - 1)) - 1.0
    qmin = -float(1 << (m + n - 1))
    q = torch.clamp(torch.round(x * scale), qmin, qmax)
    return q * (1.0 / scale)

def dyadic_rescale_tensor(acc_int32, m0, shift):
    acc_64 = acc_int32.to(torch.int64)
    scaled = (acc_64 * int(m0) + (1 << 30)) >> 31
    final_shift = shift - 31
    if final_shift > 0:
        res = (scaled + (1 << (final_shift - 1))) >> final_shift
    else:
        res = scaled << (-final_shift)
    return res.to(torch.float32)

def rns_multiply_and_crt(a, b, moduli=(7, 11, 13)):
    m_total = 1
    for m in moduli: m_total *= m
    a_int, b_int = a.to(torch.int64) % m_total, b.to(torch.int64) % m_total
    m_primes = [m_total // m for m in moduli]
    inverses = [pow(m_primes[i], -1, moduli[i]) for i in range(len(moduli))]
    c_components = [((a_int % m) * (b_int % m)) % m for m in moduli]
    reconstructed = torch.zeros_like(a_int)
    for i in range(len(moduli)):
        reconstructed += c_components[i] * (m_primes[i] * inverses[i])
    return (reconstructed % m_total).to(torch.float32)

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. Affine INT8 Parity
aff_hat, aff_q, aff_s, aff_z = affine_quantize_dequantize(weights["raw_relu"], num_bits=8)
assert torch.max(torch.abs(aff_hat - weights["affine_hat"])) < 1e-5
assert torch.max(torch.abs(aff_q - weights["affine_q"])) < 1e-5
assert abs(aff_s - float(weights["affine_s"])) < 1e-5 and abs(aff_z - float(weights["affine_z"])) < 1e-5

# 3. Symmetric INT8 Parity (Restricted & Full)
sym_restr_hat, sym_restr_q, sym_restr_s = symmetric_quantize_dequantize(weights["raw_sym"], num_bits=8, full_range=False)
assert torch.max(torch.abs(sym_restr_hat - weights["sym_hat_restr"])) < 1e-5
sym_full_hat, sym_full_q, sym_full_s = symmetric_quantize_dequantize(weights["raw_sym"], num_bits=8, full_range=True)
assert torch.max(torch.abs(sym_full_hat - weights["sym_hat_full"])) < 1e-5

# 4. Fixed-Point Q4.4 & Dyadic Rescale Parity
qmn = fixed_point_qmn(weights["raw_fp"], m=4, n=4)
assert torch.max(torch.abs(qmn - weights["qmn_hat"])) < 1e-5
dyadic_out = dyadic_rescale_tensor(weights["acc_int"].to(torch.int32), int(weights["m0"]), int(weights["shift"]))
assert torch.max(torch.abs(dyadic_out - weights["dyadic_out"])) < 1e-5

# 5. RNS Multiplication Parity
rns_out = rns_multiply_and_crt(weights["rns_a"], weights["rns_b"], moduli=(7, 11, 13))
assert torch.max(torch.abs(rns_out - weights["rns_out"])) < 1e-5
print("All Affine, Symmetric, Qm.n, Dyadic, and RNS parity assertions verified!")

In [ ]:
# Microbenchmark: Affine vs Dyadic Rescaling Latency
raw = weights["raw_relu"].repeat(10, 10)
t0 = time.perf_counter()
for _ in range(500):
    _ = affine_quantize_dequantize(raw, num_bits=8)
aff_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Affine quantization latency: {aff_us:.2f} us")

acc = weights["acc_int"].repeat(10, 10).to(torch.int32)
t0 = time.perf_counter()
for _ in range(500):
    _ = dyadic_rescale_tensor(acc, int(weights["m0"]), int(weights["shift"]))
dyadic_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Dyadic rescaling latency: {dyadic_us:.2f} us")

In [ ]:
# Pathological Misuse: Symmetric vs Affine Quantization on Post-ReLU Activations
torch.manual_seed(42)
relu_activations = torch.relu(torch.randn(1000) * 2.0 + 0.5)  # Strictly non-negative in [0, ~7]

# 1. Affine Asymmetric Quantization (Full 256 bins dedicated to [0, max])
hat_aff, _, scale_aff, _ = affine_quantize_dequantize(relu_activations, num_bits=8)
rmse_aff = float(torch.sqrt(torch.mean((relu_activations - hat_aff) ** 2)))

# 2. Symmetric Quantization (Wastes 128 bins on negative values)
hat_sym, _, scale_sym = symmetric_quantize_dequantize(relu_activations, num_bits=8, full_range=False)
rmse_sym = float(torch.sqrt(torch.mean((relu_activations - hat_sym) ** 2)))

print(f"Affine Quantization Step Size:    {scale_aff:.5f}, RMSE: {rmse_aff:.5f}")
print(f"Symmetric Quantization Step Size: {scale_sym:.5f}, RMSE: {rmse_sym:.5f}")
print(f"Step size ratio: {scale_sym / scale_aff:.2f}x (Symmetric has ~2x wider steps due to wasted negative bins)")
assert scale_sym > scale_aff * 1.8, "Symmetric step size must be ~2x larger than Affine on one-sided ReLU data!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Distribution Alignment**: Use Affine Quantization for asymmetric/one-sided activations (ReLU/SiLU) to utilize 100% of quantization dynamic range; use Symmetric Quantization for zero-centered weights and gradients.
2. **Dyadic Rescaling**: Floating-point scale factors $M = (s_W s_X) / s_Y$ are decomposed into integer multipliers $M_0$ and right-shift exponents $n$, eliminating floating-point hardware from inference.
3. **RNS Parallelism**: Residue Number Systems eliminate carry propagation delays across wide registers by performing parallel arithmetic over small coprime moduli.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.md](../../../markdown/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.md)
- **PyTorch 2.x Module:** [01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.py](../../../pytorch2/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.py)
- **Functional JAX Module:** [01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.py](../../../jax/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.py)
- **Native Rust Candle Track:** [01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.rs](../../../candle/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.rs)
- **Hardware Kernel (CPU / WGSL):** [01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.rs](../../../kernels/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.rs)
- **Golden SafeTensors:** [01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.safetensors](../../../weights/14_quantisation/01_foundations_formats_and_observers/01_int_uint_fixed_point_q_format_dyadic_and_rns_arithmetic.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Stochastic Rounding and Master Weight Free BF16 Training](../../13_precision_and_numerics/06_stochastic_rounding_and_master_weight_free_bf16_training.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 02. Logarithmic PoT APoT Mu-Law and Posit Quantization](02_logarithmic_pot_apot_mulaw_and_posit_quantization.ipynb) &rarr;